In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings

warnings.filterwarnings('ignore')

#_____Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#_____Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format','{:.2f}'.format)

print("✓ Libraries Loaded")

✓ Libraries Loaded


In [2]:
#_____Constants

N_ROWS = 1000

REGIONS = ['North', 'South', 'East', 'West']
PRODUCTS = ['Software Liscense', 'Consulting', 'Support Package', 'Hardware', 'Training']
SALES_REPS = ['Alice', 'Bob', 'Carol', 'David', 'Eva', 'Frank', 'Grace', 'Henry']
DEAL_STAGES = ['Closed Won', 'Closed Lost']

print("✓ Constants defined")
print(f" → {len(SALES_REPS)} sales reps, {len(PRODUCTS)} products, {len(REGIONS)} regions")

✓ Constants defined
 → 8 sales reps, 5 products, 4 regions


In [3]:
# _____ Data Generation

start_date = datetime(2023, 1, 1)
dates = [start_date + timedelta(days=random.randint(0,364)) for _ in range(N_ROWS)]

# _____ Build the DataFrame

df = pd.DataFrame({
    'date':    dates,
    'region':  np.random.choice(REGIONS, N_ROWS),
    'sales_rep': np.random.choice(SALES_REPS, N_ROWS),
    'product':   np.random.choice(PRODUCTS, N_ROWS),

    'deal_stage': np.random.choice(DEAL_STAGES, N_ROWS, p=[0.65, 0.35]),

    'deal_size':  np.random.choice(['Small', 'Medium', 'Large'], N_ROWS, p=[0.5, 0.35, 0.15]),
    'units_sold': np.random.randint(1, 20, N_ROWS),
    'unit_price': np.random.uniform(500, 15000, N_ROWS).round(2),
})

# _____ Revenue Column
df['revenue'] = np.where(
    df['deal_stage'] == 'Closed Won',
    (df['units_sold'] * df['unit_price']).round(2),
    0
)

print(f"✓ Dataset created: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(10)


✓ Dataset created: 1,000 rows x 9 columns


,date,region,sales_rep,product,deal_stage,deal_size,units_sold,unit_price,revenue
0,2023-11-24,East,Frank,Hardware,Closed Won,Medium,11,14309.61,157405.71
1,2023-02-27,West,Grace,Support Package,Closed Won,Small,5,5267.93,26339.65
2,2023-01-13,North,Alice,Training,Closed Won,Medium,13,10103.55,131346.15
3,2023-05-21,East,Alice,Software Liscense,Closed Won,Small,5,11401.46,57007.30
4,2023-05-06,East,Alice,Training,Closed Lost,Medium,13,12273.22,0.00
5,2023-04-25,West,Eva,Software Liscense,Closed Won,Small,6,14213.37,85280.22
6,2023-03-13,North,David,Consulting,Closed Lost,Small,5,5037.33,0.00
7,2023-02-22,North,Eva,Software Liscense,Closed Won,Large,9,12948.34,116535.06
8,2023-12-13,East,David,Hardware,Closed Lost,Medium,16,2409.46,0.00
9,2023-10-07,South,Henry,Support Package,Closed Won,Small,2,10723.09,21446.18


In [4]:
print("=== SHAPE ===")
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns\n")

print("=== COLUMN TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== BASIC STATS ===")
df.describe()

=== SHAPE ===
1,000 rows, 9 columns

=== COLUMN TYPES ===
date          datetime64[us]
region                   str
sales_rep                str
product                  str
deal_stage               str
deal_size                str
units_sold             int64
unit_price           float64
revenue              float64
dtype: object

=== MISSING VALUES ===
date          0
region        0
sales_rep     0
product       0
deal_stage    0
deal_size     0
units_sold    0
unit_price    0
revenue       0
dtype: int64

=== BASIC STATS ===


,date,units_sold,unit_price,revenue
count,1000,1000.00,1000.00,1000.00
mean,2023-07-02 13:30:43.200000,10.37,7690.84,50622.87
min,2023-01-01 00:00:00,1.00,503.29,0.00
25%,2023-04-08 00:00:00,6.00,4148.53,0.00
50%,2023-06-29 00:00:00,11.00,7422.67,23419.90
75%,2023-10-04 00:00:00,15.00,11303.68,81750.34
max,2023-12-31 00:00:00,19.00,14982.50,278086.09
std,NaN,5.35,4186.24,63242.22


In [5]:
df.to_csv('data/raw_sales_data.csv' , index=False)
print(" Saved to data/raw_sales_data.csv")

 Saved to data/raw_sales_data.csv


# ════════════════════════════════════════════════════════════
# PHASE 2: DATA CLEANING & FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════

In [6]:
#Always work from a copy of the raw data - never modify the original
df = pd.read_csv('data/raw_sales_data.csv' , parse_dates=['date'])

#__Introduce realistice "dirty data" problems__
dirty = df.copy()

# 1. Inject missing values (-5% of rows in key columns)
null_indices = np.random.choice(dirty.index, size=50, replace= False)
dirty.loc[null_indices, 'unit_price'] = np.nan

null_indices2 = np.random.choice(dirty.index, size=30, replace = False)
dirty.loc[null_indices2, 'region'] = np.nan

# 2. Inject duplicates (same row appearing twice)
duplicate_rows = dirty.sample(20, random_state = 42)
dirty = pd.concat([dirty, duplicate_rows], ignore_index=True)

# 3. Mess up casing in a text column
dirty['product'] = dirty['product'].str.lower() # all lowercase now
dirty.loc[np.random.choice(dirty.index, 30), 'product'] = '  software license  ' #extra spaces

# 4. Store a number as text (common Excel import problem)
dirty['units_sold'] = dirty['units_sold'].astype(str)

print(f"Dirty dataset:      {dirty.shape[0]} rows (was 1,000)")
print(f"Missing unit_price: {dirty['unit_price'].isnull().sum()}")
print(f"Missing region:     {dirty['region'].isnull().sum()}")
print(f"Duplicate rows:     {dirty.duplicated().sum()}")
print(f"units_sold dtype:   {dirty['units_sold'].dtype} <- should be int, not object!")

Dirty dataset:      1020 rows (was 1,000)
Missing unit_price: 52
Missing region:     31
Duplicate rows:     18
units_sold dtype:   str <- should be int, not object!


In [7]:
#Remove duplicates
#Always do this FIRST - duplicates inflate every metric
before = len(dirty)
clean = dirty.drop_duplicates()
after = len(clean)

print(f" Removed {before - after} duplicate rows ({before} -> {after})")

 Removed 18 duplicate rows (1020 -> 1002)


In [8]:
#Handle missing values

#Missing unit_price -> fill with the MEDIAN (not mean - median is
#resistant to outliers, which matter a lot in pricing data)
median_price = clean['unit_price'].median()
clean['unit_price'] = clean['unit_price'].fillna(median_price)
print(f" Filled missing unit_price with median: ${median_price:,.2f}")

#Missing region -> fill with 'Unkown'
#(don't guess a region - flag it honestly)
clean['region'] = clean['region'].fillna('Unknown')
print(f" Filled missing region with 'Unkown'")

#Verify - should be all zeros now
print(f"\n  Remaining nulls:")
print(clean[['unit_price', 'region']].isnull().sum())

 Filled missing unit_price with median: $7,411.74
 Filled missing region with 'Unkown'

  Remaining nulls:
unit_price    0
region        0
dtype: int64


In [10]:
# Fix data types

#Convert units_sold back from string to integer
clean['units_sold'] = pd.to_numeric(clean['units_sold'], errors='coerce').astype(int)
print(f" units_sold dtype fixed: {clean['units_sold'].dtype}")

# .str.strip() removes leading/trailing spaces
# .str.title() makes "software liscence -> "Software License"
clean['product'] = clean['product'].str.strip().str.title()
clean['region'] = clean['region'].str.strip().str.title()
clean['sales_rep'] = clean['sales_rep'].str.strip().str.title()

print(f" Text columns standardized")
print(f"\n Unique products: {sorted(clean['product'].unique())}")

 units_sold dtype fixed: int64
 Text columns standardized

 Unique products: ['Consulting', 'Hardware', 'Software License', 'Software Liscense', 'Support Package', 'Training']


In [19]:
#__Feature Engineering
# Creating new columns that answer business questions

# 1. Extract time components that answer business questions
clean['month']    = clean['date'].dt.month
clean['month_name'] = clean['date'].dt.strftime('%b')  #Jan, Feb, Mar...
clean['quarter']    = clean['date'].dt.quarter         #Q1, Q2, Q3, Q4
clean['day_of_week'] = clean['date'].dt.day_name()     # Monday, Tuesday..

# 2. Win/Loss as a number (useful for calculating win rates)
clean['is_won'] = (clean['deal_stage'] == 'Closed Won').astype(int)
#True -> 1, False -> 0

# 3. Recalculate revenue with the clean data
clean['revenue'] = np.where(
    clean['deal_stage'] == 'Closed Won',
    (clean['units_sold'] * clean['unit_price']).round(2),
    0
)

# 4. Revenue per unit (useful for comparing product profitability)
clean['revenue_per_unit'] = np.where (
    clean['units_sold'] > 0,
    (clean['revenue'] / clean['units_sold']).round(2),
    0
)
print(" New columns created:")
new_cols = ['month', 'month_name', 'quarter', 'day_of_week', 'is_won', 'revenue_per_unit']
for col in new_cols:
    print(f" -> {col}: {clean[col].dtype}")

clean.head(3)

 New columns created:
 -> month: int32
 -> month_name: str
 -> quarter: int32
 -> day_of_week: str
 -> is_won: int64
 -> revenue_per_unit: float64


,date,region,sales_rep,product,deal_stage,deal_size,units_sold,unit_price,revenue,month,month_name,quarter,day_of_week,is_won,revenue_per_unit
0,2023-11-24,East,Frank,Hardware,Closed Won,Medium,11,14309.61,157405.71,11,Nov,4,Friday,1,14309.61
1,2023-02-27,West,Grace,Support Package,Closed Won,Small,5,5267.93,26339.65,2,Feb,1,Monday,1,5267.93
2,2023-01-13,North,Alice,Training,Closed Won,Medium,13,10103.55,131346.15,1,Jan,1,Friday,1,10103.55


In [25]:
# Final data quality check ___
print("=== CLEAN DATASET SUMMARY ===")
print(f"Shape:     {clean.shape[0]:,} rows x {clean.shape[1]} columns")
print(f"Missing values: {clean.isnull().sum().sum()} total")
print(f"Duplicates:     {clean.duplicated().sum()}")
print(f"Date range:     {clean['date'].min().date()} -> {clean['date'].max().date()}")
print(f"Total revenue:  ${clean['revenue'].sum():,.2f}")
print(f"Win rate:       {clean['is_won'].mean():.1%}")

# Save the clean version - always keeep raw and clean separate
clean.to_csv('data/clean_sales_data.csv', index = False)
print("\n Clean data saved to data/clean_sales_data.csv")


=== CLEAN DATASET SUMMARY ===
Shape:     1,002 rows x 15 columns
Missing values: 0 total
Duplicates:     0
Date range:     2023-01-01 -> 2023-12-31
Total revenue:  $50,681,867.30
Win rate:       63.6%

 Clean data saved to data/clean_sales_data.csv
